In [2]:
import torch

print(f"Torch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")
    print(f"Current Device: {torch.cuda.current_device()}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")
    print(f"Memory Reserved: {torch.cuda.memory_reserved(0)/1024**3:.2f} GB")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Torch version: 2.5.1+cu121
CUDA Available: True
GPU: NVIDIA RTX A6000
GPU Count: 1
Current Device: 0
Memory Allocated: 0.00 GB
Memory Reserved: 0.00 GB
Using device: cuda


In [3]:
import json
import pandas as pd

def load_jsonl(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

df = load_jsonl("Train_validation_Merged_ASQE_N4000.jsonl")

print("Original shape:", df.shape)
df.head()

Original shape: (4000, 2)


,text,labels
0,He is not helpful at all and makes fun of stud...,"[{'aspect': 'He', 'opinion': 'Very rude', 'pol..."
1,"This course was amazing. For the first time, t...","[{'aspect': 'This course', 'opinion': 'I loved..."
2,Exams were very easy and he allowed us to do e...,"[{'aspect': 'Exams', 'opinion': 'very easy', '..."
3,The lecturers from Exeter have helped me throu...,"[{'aspect': 'lecturers', 'opinion': 'helped me..."
4,"Extremely boring and lectures were useless, bu...","[{'aspect': 'lectures', 'opinion': 'Extremely ..."


In [4]:
def get_domain(categories):
    prefixes = [cat.split("#")[0] for cat in categories]

    if "UNIVERSITY" in prefixes:
        return "university"
    elif "STAFF" in prefixes:
        return "teacher"
    elif "COURSE" in prefixes:
        return "course"
    else:
        return "unknown"

df["categories"] = df["labels"].apply(lambda x: [l["category"] for l in x])
df["domain"] = df["categories"].apply(get_domain)

print(df["domain"].value_counts())

domain
teacher       2058
course        1553
university     327
unknown         62
Name: count, dtype: int64


In [5]:
# Remove unknown domain
df = df[df["domain"] != "unknown"]

print("After removing unknown:")
print(df["domain"].value_counts())

After removing unknown:
domain
teacher       2058
course        1553
university     327
Name: count, dtype: int64


In [6]:
def explode_aspects(df):
    rows = []

    for _, row in df.iterrows():
        review = row["text"]
        domain = row["domain"]

        for label in row["labels"]:
            aspect = str(label["aspect"]).strip()
            sentiment = str(label["polarity"]).strip().lower()

            if aspect == "null":
                continue
            if sentiment not in ["positive", "negative", "neutral"]:
                continue

            rows.append({
                "text": review,
                "aspect": aspect,
                "label": sentiment,
                "domain": domain
            })

    return pd.DataFrame(rows)

df = explode_aspects(df)

print("After explosion:", df.shape)
df.head()

After explosion: (14961, 4)


,text,aspect,label,domain
0,He is not helpful at all and makes fun of stud...,He,negative,teacher
1,He is not helpful at all and makes fun of stud...,He,negative,teacher
2,He is not helpful at all and makes fun of stud...,He,negative,teacher
3,"This course was amazing. For the first time, t...",This course,positive,course
4,"This course was amazing. For the first time, t...",This course,positive,course


In [7]:
import re  

bad_aspects = ["he", "she", "it", "they", "him", "her", "them"]

df = df[~df["aspect"].str.lower().isin(bad_aspects)]
df = df[df["aspect"].str.len() > 2]

df["aspect"] = df["aspect"].str.lower().str.strip()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["text"].apply(clean_text)

print("After cleaning:", df.shape)

After cleaning: (13579, 4)


In [8]:
print(df["label"].value_counts())

label
positive    7979
negative    4196
neutral     1404
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

print("\nSplit sizes:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


Split sizes:
Train: 10863
Validation: 1358
Test: 1358


In [10]:
test_df.to_csv("shared_test_set.csv", index=False)

In [11]:
import wandb

wandb.login()

wandb.init(
    project="absa-multidomain",
    name="llama-lora-multidomain"
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/jovyan/.netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: Currently logged in as: sumukhssagar (sumukhssagar-dublin-city-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [12]:
def format_instruction(row):
    return f"""<s>[INST] You are an expert sentiment classifier.

Classify sentiment into:
positive, negative, or neutral.

Domain: {row['domain']}

Review:
{row['text']}

Aspect:
{row['aspect']}

Answer:
[/INST] {row['label']}"""

In [13]:
train_df["text"] = train_df.apply(format_instruction, axis=1)
val_df["text"] = val_df.apply(format_instruction, axis=1)
test_df["text"] = test_df.apply(format_instruction, axis=1)

print(train_df["text"].iloc[0])

<s>[INST] You are an expert sentiment classifier.

Classify sentiment into:
positive, negative, or neutral.

Domain: course

Review:
way too broad of a course and doesn't need to exist.

Aspect:
course

Answer:
[/INST] negative


In [14]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"


tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=HF_TOKEN   
)
tokenizer.pad_token = tokenizer.eos_token



bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)



model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN   
)



model = prepare_model_for_kbit_training(model)



lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)



model = get_peft_model(model, lora_config)



model.config.use_cache = False
model.gradient_checkpointing_enable()
model.train()

print("Model + LoRA loaded successfully")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model + LoRA loaded successfully


In [15]:
def tokenize_function(example):
    full_text = example["text"]

    parts = full_text.split("[/INST]")

    if len(parts) < 2:
        return {
            "input_ids": [0]*128,
            "attention_mask": [0]*128,
            "labels": [-100]*128
        }

    prompt = parts[0] + "[/INST]"
    answer = parts[-1].strip()   

    prompt_ids = tokenizer(prompt, truncation=True, max_length=128)["input_ids"]
    answer_ids = tokenizer(answer, truncation=True, max_length=128)["input_ids"]

    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids

    max_len = 128
    input_ids = input_ids[:max_len]
    labels = labels[:max_len]

    padding_length = max_len - len(input_ids)

    input_ids += [tokenizer.pad_token_id] * padding_length
    labels += [-100] * padding_length

    attention_mask = [1 if id != tokenizer.pad_token_id else 0 for id in input_ids]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [16]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["text"]])
val_dataset = Dataset.from_pandas(val_df[["text"]])

In [17]:
train_dataset = train_dataset.map(tokenize_function, remove_columns=["text"])
val_dataset = val_dataset.map(tokenize_function, remove_columns=["text"])

Map:   0%|          | 0/10863 [00:00<?, ? examples/s]

Map:   0%|          | 0/1358 [00:00<?, ? examples/s]

In [18]:
train_dataset.set_format(type="torch")
val_dataset.set_format(type="torch")

In [21]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./llama_lora_multi",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=3e-5,
    logging_steps=20,
    save_strategy="no",
    eval_strategy="epoch",   
    report_to="none",
    fp16=True,
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.222798,0.206469
2,0.202662,0.190864
3,0.158216,0.210748


TrainOutput(global_step=4074, training_loss=0.2939671160078072, metrics={'train_runtime': 13567.5127, 'train_samples_per_second': 2.402, 'train_steps_per_second': 0.3, 'total_flos': 1.8800648326427443e+17, 'train_loss': 0.2939671160078072, 'epoch': 3.0})

In [26]:
def predict(df, sample_size=1000):
    import torch

    df = df.sample(n=min(sample_size, len(df)), random_state=42).reset_index(drop=True)

    model.eval()
    preds, labels = [], []

    device = next(model.parameters()).device

    for i in range(len(df)):
        review = df.iloc[i]["text"]
        aspect = df.iloc[i]["aspect"]
        label = df.iloc[i]["label"]

        prompt = f"""<s>[INST] Classify the sentiment.

Review:
{review}

Aspect:
{aspect}

Answer:
[/INST]"""

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=3,
                max_length=None,   
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )

        decoded = tokenizer.decode(
            output[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        response = decoded.split("[/INST]")[-1].strip().lower()

        if "positive" in response:
            pred = "positive"
        elif "negative" in response:
            pred = "negative"
        elif "neutral" in response:
            pred = "neutral"
        else:
            pred = "neutral"

        preds.append(pred)
        labels.append(label)

    return preds, labels

In [27]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("\n===== TRAIN RESULTS =====")

train_sample = train_df.sample(n=1000, random_state=42)

preds_train, labels_train = predict(train_sample)

print("Accuracy:", accuracy_score(labels_train, preds_train))
print("F1 Macro:", f1_score(labels_train, preds_train, average="macro"))
print("F1 Weighted:", f1_score(labels_train, preds_train, average="weighted"))

print("\nClassification Report:\n")
print(classification_report(labels_train, preds_train))


===== TRAIN RESULTS =====
Accuracy: 0.863
F1 Macro: 0.789208115678704
F1 Weighted: 0.8623662162162163

Classification Report:

              precision    recall  f1-score   support

    negative       0.90      0.84      0.87       308
     neutral       0.59      0.59      0.59       111
    positive       0.90      0.93      0.91       581

    accuracy                           0.86      1000
   macro avg       0.80      0.78      0.79      1000
weighted avg       0.86      0.86      0.86      1000



In [28]:
print("\n===== TEST RESULTS (MULTI-DOMAIN) =====")

test_sample = test_df.sample(n=1000, random_state=42)

preds_test, labels_test = predict(test_sample)

print("Accuracy:", accuracy_score(labels_test, preds_test))
print("F1 Macro:", f1_score(labels_test, preds_test, average="macro"))
print("F1 Weighted:", f1_score(labels_test, preds_test, average="weighted"))

print("\nClassification Report:\n")
print(classification_report(labels_test, preds_test))


===== TEST RESULTS (MULTI-DOMAIN) =====
Accuracy: 0.834
F1 Macro: 0.7474779596483451
F1 Weighted: 0.835982859421399

Classification Report:

              precision    recall  f1-score   support

    negative       0.85      0.81      0.83       312
     neutral       0.49      0.55      0.52       104
    positive       0.89      0.90      0.90       584

    accuracy                           0.83      1000
   macro avg       0.74      0.75      0.75      1000
weighted avg       0.84      0.83      0.84      1000



In [29]:
import torch
import gc


del model
del trainer


gc.collect()
torch.cuda.empty_cache()

print("Old model cleared from memory")

Old model cleared from memory


In [33]:
def format_instruction_fair(row):
    return f"""<s>[INST] You are an expert sentiment classifier.

Classify sentiment into:
positive, negative, or neutral.

Review:
{row['text']}

Aspect:
{row['aspect']}

Answer:
[/INST] {row['label']}"""

In [34]:
train_df_fair, temp_df_fair = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

val_df_fair, test_df_fair = train_test_split(
    temp_df_fair,
    test_size=0.5,
    stratify=temp_df_fair["label"],
    random_state=42
)

In [35]:
train_df_fair["text"] = train_df_fair.apply(format_instruction_fair, axis=1)
val_df_fair["text"] = val_df_fair.apply(format_instruction_fair, axis=1)
test_df_fair["text"] = test_df_fair.apply(format_instruction_fair, axis=1)

In [36]:
print(train_df_fair["text"].iloc[0])

<s>[INST] You are an expert sentiment classifier.

Classify sentiment into:
positive, negative, or neutral.

Review:
way too broad of a course and doesn't need to exist.

Aspect:
course

Answer:
[/INST] negative


In [37]:
from datasets import Dataset

train_dataset_fair = Dataset.from_pandas(train_df_fair[["text"]])
val_dataset_fair = Dataset.from_pandas(val_df_fair[["text"]])

In [38]:
train_dataset_fair = train_dataset_fair.map(tokenize_function, remove_columns=["text"])
val_dataset_fair = val_dataset_fair.map(tokenize_function, remove_columns=["text"])

train_dataset_fair.set_format(type="torch")
val_dataset_fair.set_format(type="torch")

Map:   0%|          | 0/10863 [00:00<?, ? examples/s]

Map:   0%|          | 0/1358 [00:00<?, ? examples/s]

In [39]:
model_fair = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)

model_fair = prepare_model_for_kbit_training(model_fair)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [40]:
lora_config_fair = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model_fair = get_peft_model(model_fair, lora_config_fair)

model_fair.config.use_cache = False
model_fair.gradient_checkpointing_enable()
model_fair.train()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

In [41]:
training_args_fair = TrainingArguments(
    output_dir="./llama_lora_multi_fair",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=3e-5,
    logging_steps=20,
    save_strategy="no",
    eval_strategy="epoch",
    report_to="none",
    fp16=True,
    remove_unused_columns=False
)

trainer_fair = Trainer(
    model=model_fair,
    args=training_args_fair,
    train_dataset=train_dataset_fair,
    eval_dataset=val_dataset_fair
)

trainer_fair.train()

Epoch,Training Loss,Validation Loss
1,0.218597,0.206319
2,0.192631,0.198511
3,0.169711,0.211536


TrainOutput(global_step=4074, training_loss=0.2912947508887327, metrics={'train_runtime': 13544.5101, 'train_samples_per_second': 2.406, 'train_steps_per_second': 0.301, 'total_flos': 1.8800648326427443e+17, 'train_loss': 0.2912947508887327, 'epoch': 3.0})

In [42]:
def predict_fair(df, sample_size=1000):
    import torch

    df = df.sample(n=min(sample_size, len(df)), random_state=42).reset_index(drop=True)

    model_fair.eval()
    preds, labels = [], []

    device = next(model_fair.parameters()).device

    for i in range(len(df)):
        review = df.iloc[i]["text"]
        aspect = df.iloc[i]["aspect"]
        label = df.iloc[i]["label"]

        prompt = f"""<s>[INST] Classify the sentiment.

Review:
{review}

Aspect:
{aspect}

Answer:
[/INST]"""

        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            output = model_fair.generate(
                **inputs,
                max_new_tokens=3,
                max_length=None,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )

        decoded = tokenizer.decode(
            output[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        response = decoded.split("[/INST]")[-1].strip().lower()

        if "positive" in response:
            pred = "positive"
        elif "negative" in response:
            pred = "negative"
        elif "neutral" in response:
            pred = "neutral"
        else:
            pred = "neutral"

        preds.append(pred)
        labels.append(label)

    return preds, labels

In [43]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("\n===== TRAIN RESULTS (FAIR MULTI-DOMAIN) =====")

train_sample = train_df_fair.sample(n=1000, random_state=42)

preds_train, labels_train = predict_fair(train_sample)

print("Accuracy:", accuracy_score(labels_train, preds_train))
print("F1 Macro:", f1_score(labels_train, preds_train, average="macro"))
print("F1 Weighted:", f1_score(labels_train, preds_train, average="weighted"))

print("\nClassification Report:\n")
print(classification_report(labels_train, preds_train))


===== TRAIN RESULTS (FAIR MULTI-DOMAIN) =====
Accuracy: 0.808
F1 Macro: 0.7212414391941766
F1 Weighted: 0.8158325904503552

Classification Report:

              precision    recall  f1-score   support

    negative       0.89      0.75      0.82       308
     neutral       0.40      0.56      0.46       111
    positive       0.88      0.88      0.88       581

    accuracy                           0.81      1000
   macro avg       0.72      0.73      0.72      1000
weighted avg       0.83      0.81      0.82      1000



In [44]:
print("\n===== TEST RESULTS (FAIR MULTI-DOMAIN) =====")

test_sample = test_df_fair.sample(n=1000, random_state=42)

preds_test, labels_test = predict_fair(test_sample)

print("Accuracy:", accuracy_score(labels_test, preds_test))
print("F1 Macro:", f1_score(labels_test, preds_test, average="macro"))
print("F1 Weighted:", f1_score(labels_test, preds_test, average="weighted"))

print("\nClassification Report:\n")
print(classification_report(labels_test, preds_test))


===== TEST RESULTS (FAIR MULTI-DOMAIN) =====
Accuracy: 0.789
F1 Macro: 0.6947648454678741
F1 Weighted: 0.8001626074536404

Classification Report:

              precision    recall  f1-score   support

    negative       0.86      0.73      0.79       312
     neutral       0.35      0.54      0.42       104
    positive       0.88      0.86      0.87       584

    accuracy                           0.79      1000
   macro avg       0.70      0.71      0.69      1000
weighted avg       0.82      0.79      0.80      1000

